In [3]:
import pandas as pd

In [20]:
file_path = "results/results__test__001.jsonl"

In [21]:
df = pd.read_json(file_path, lines=True)

# Extract token counts from the nested structure
df['input_tokens'] = df['_meta'].apply(lambda x: x["token_count"]['input_tokens'])
df['output_tokens'] = df['_meta'].apply(lambda x: x["token_count"]['output_tokens'])

# Group by agent_id and sum tokens
token_sums = df.groupby('agent_id').agg({
    'input_tokens': 'sum',
    'output_tokens': 'sum'
}).reset_index()
token_sums

,agent_id,input_tokens,output_tokens
0,baseline_agentic,3585,405
1,baseline_answerable,999,41
2,baseline_naive,932,193
3,baseline_pretraining,103,42
4,student,15492,1743


In [22]:
def cost_estimation(tokens, prizes_per_mil):
    total_cost = tokens * prizes_per_mil / 10**6
    return total_cost

def cost_estimate_input(input_tokens):
    return cost_estimation(int(input_tokens), 0.8)

def cost_estimate_output(output_tokens):
    return cost_estimation(int(output_tokens), 4)


In [23]:
token_sums['input_tokens']

0     3585
1      999
2      932
3      103
4    15492
Name: input_tokens, dtype: int64

In [24]:
# Calculate costs for each group
token_sums['input_cost'] = token_sums['input_tokens'].apply(cost_estimate_input)
token_sums['output_cost'] = token_sums['output_tokens'].apply(cost_estimate_output)
token_sums['total_cost'] = token_sums['input_cost'] + token_sums['output_cost']

In [25]:
token_sums

,agent_id,input_tokens,output_tokens,input_cost,output_cost,total_cost
0,baseline_agentic,3585,405,0.002868,0.001620,0.004488
1,baseline_answerable,999,41,0.000799,0.000164,0.000963
2,baseline_naive,932,193,0.000746,0.000772,0.001518
3,baseline_pretraining,103,42,0.000082,0.000168,0.000250
4,student,15492,1743,0.012394,0.006972,0.019366
